# Milestone 2 — Feature Engineering, TF-IDF and Content-Based Recommendation

### Objective
Convert TED Talk content into numerical TF-IDF features, compute cosine similarity, and demonstrate a top-N content-based recommendation engine.

### Method choice
TF-IDF is selected as a transparent and reproducible baseline because the available dataset includes talk titles, topics, descriptions and transcripts. No per-user ratings or watch-history records are available, so this notebook implements content-based filtering rather than claiming collaborative filtering.

## 1. Load data and prepare content

In [1]:
from pathlib import Path
import re, string, joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

DATA_PATH = Path("ted_talks_final_cleaned.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place ted_talks_final_cleaned.csv beside this notebook or update DATA_PATH.")
df = pd.read_csv(DATA_PATH, low_memory=False)
required = ["talk_id", "title", "speaker_1", "topics", "description", "transcript"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing from dataset: {missing}")

df = df.drop_duplicates(subset=["talk_id"]).reset_index(drop=True)
for col in ["title", "speaker_1", "topics", "description", "transcript"]:
    df[col] = df[col].fillna("").astype(str)

def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

# Weight title and topic information by repeating them once in the content string.
df["recommendation_text"] = (
    df["title"] + " " + df["title"] + " " +
    df["topics"] + " " + df["topics"] + " " +
    df["description"] + " " + df["transcript"]
).map(normalize_text)

print("Dataset ready:", df.shape)
display(df[["talk_id", "title", "speaker_1", "topics"]].head())

Dataset ready: (4005, 33)


,talk_id,title,speaker_1,topics
0,1,averting the climate crisis,al gore,alternative energy cars climate change culture...
1,92,the best stats youve ever seen,hans rosling,Africa Asia Google demo economics global issue...
2,7,simplicity sells,david pogue,computers entertainment interface design media...
3,53,greening the ghetto,majora carter,MacArthur grant activism business cities envir...
4,66,do schools kill creativity,sir ken robinson,children creativity culture dance education pa...


## 2. TF-IDF vectorization

In [2]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,
    dtype=np.float32
)
tfidf_matrix = tfidf.fit_transform(df["recommendation_text"])
print("TF-IDF matrix created successfully.")
print("Matrix shape (talks, features):", tfidf_matrix.shape)
print("Non-zero feature values:", tfidf_matrix.nnz)
print("Sparsity:", round(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]), 6))

TF-IDF matrix created successfully.
Matrix shape (talks, features): (4005, 50000)
Non-zero feature values: 2232022
Sparsity: 0.988854


## 3. Similarity-based recommendation function

In [3]:
talk_id_to_index = {int(tid): idx for idx, tid in enumerate(df["talk_id"])}

def recommend_by_talk_id(talk_id, top_n=5):
    if int(talk_id) not in talk_id_to_index:
        raise KeyError(f"talk_id {talk_id} is not present in the catalogue.")
    query_idx = talk_id_to_index[int(talk_id)]
    scores = cosine_similarity(tfidf_matrix[query_idx], tfidf_matrix).ravel()
    scores[query_idx] = -1.0  # exclude the input talk itself
    top_idx = np.argsort(scores)[::-1][:top_n]
    result_cols = [c for c in ["talk_id", "title", "speaker_1", "topics", "url"] if c in df.columns]
    result = df.iloc[top_idx][result_cols].copy()
    result.insert(0, "similarity_score", scores[top_idx].round(4))
    return result.reset_index(drop=True)

# Reproducible demonstration using an existing catalogue record.
sample_talk_id = 1
seed = df.loc[talk_id_to_index[sample_talk_id]]
print("INPUT TALK")
print("Title:", seed["title"])
print("Speaker:", seed["speaker_1"])
print("Topics:", seed["topics"])
print("\nTOP 5 CONTENT-BASED RECOMMENDATIONS")
recommendations = recommend_by_talk_id(sample_talk_id, top_n=5)
display(recommendations)

INPUT TALK
Title: averting the climate crisis
Speaker: al gore
Topics: alternative energy cars climate change culture environment global issues science sustainability technology

TOP 5 CONTENT-BASED RECOMMENDATIONS


,similarity_score,talk_id,title,speaker_1,topics,url
0,0.1630,243,new thinking on the climate crisis,al gore,activism climate change global issues science,https://www.ted.com/talks/al_gore_new_thinking...
1,0.1608,535,what comes after an inconvenient truth,al gore,alternative energy climate change energy envir...,https://www.ted.com/talks/al_gore_what_comes_a...
2,0.1520,2441,the case for optimism on climate change,al gore,climate change weather science alternative ene...,https://www.ted.com/talks/al_gore_the_case_for...
3,0.1483,54715,how we can turn the tide on climate,chris anderson,climate change environment activism sustainabi...,https://www.ted.com/talks/christiana_figueres_...
4,0.1330,128,salvation and profit in greentech,john doerr,climate change environment global issues green...,https://www.ted.com/talks/john_doerr_salvation...


## 4. Inspect the most similar talks and score interpretation

In [4]:
# Similarity scores are cosine similarities between TF-IDF vectors.
# They indicate content overlap, not a probability that a user will like a talk.
print("Similarity score range:", float(recommendations["similarity_score"].min()), "to", float(recommendations["similarity_score"].max()))
print("Input talk excluded:", int(sample_talk_id) not in recommendations["talk_id"].astype(int).tolist())

Similarity score range: 0.13300000131130219 to 0.16300000250339508
Input talk excluded: True


## 5. Save reusable model artifacts

In [5]:
MODEL_DIR = Path("models_milestone2")
OUTPUT_DIR = Path("outputs_milestone2")
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(tfidf, MODEL_DIR / "ted_talks_tfidf_vectorizer.joblib")
sparse.save_npz(MODEL_DIR / "ted_talks_tfidf_matrix.npz", tfidf_matrix)
df[["talk_id", "title", "speaker_1", "topics", "url"]].to_csv(
    MODEL_DIR / "ted_talks_catalogue_for_recommendation.csv", index=False
)
recommendations.to_csv(OUTPUT_DIR / "sample_recommendations_talk_id_1.csv", index=False)

artifact_report = pd.DataFrame([
    ["TF-IDF vectorizer", "models_milestone2/ted_talks_tfidf_vectorizer.joblib"],
    ["TF-IDF sparse matrix", "models_milestone2/ted_talks_tfidf_matrix.npz"],
    ["Recommendation catalogue", "models_milestone2/ted_talks_catalogue_for_recommendation.csv"],
    ["Sample recommendation output", "outputs_milestone2/sample_recommendations_talk_id_1.csv"]
], columns=["artifact", "saved_path"])
display(artifact_report)

,artifact,saved_path
0,TF-IDF vectorizer,models_milestone2/ted_talks_tfidf_vectorizer.j...
1,TF-IDF sparse matrix,models_milestone2/ted_talks_tfidf_matrix.npz
2,Recommendation catalogue,models_milestone2/ted_talks_catalogue_for_reco...
3,Sample recommendation output,outputs_milestone2/sample_recommendations_talk...
